In [1]:
%%time
from datetime import datetime
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.types import (StructType, StructField, StringType, 
    FloatType, DateType, ArrayType, DoubleType, BooleanType)
from pyspark.sql.functions import col,array_contains

CPU times: user 9.22 ms, sys: 985 µs, total: 10.2 ms
Wall time: 8.79 ms


In [2]:
# this PySpark Notebook is intended to prototype the workflow of loading a CSV file
# stored in a GCP bucket into a staging-area BigQuery table
# Note: if such a table exists, it will be overwritten with the new data from CSV file

In [3]:
%%time
def load_csv_to_bigquery(
    csv_file: str,
    gcp_project_id: str,
    dataset_name: str,
    table_name: str,
    temp_gcp_bucket_name: str
):
    """
    Read the csv file from GCP,
    then uploads it to a target table in BigQuery.
    
    csv_file - either a fully qualified path or Google Cloud Storate uri of a csv file
    gcp_project_id - the google cloud project ID
    dataset_name - the name of the target dataset (database) in BigQuery
    table_name - the name of the target table in the BigQuery dataset 
                Note: if such a table exists, it will be overwritten with the new data from CSV file
    temp_gcp_bucket_name - the name of GCP bucket (wihtout uri prefix) used as a temp space for Spark to
                           write the data into BigQuery
    """
    
    # read the CSV file into memory
    df = spark.read.option("header", True) \
        .option("quote", "\"") \
        .option("escape", "\"") \
        .csv(csv_file)
    
    # Replaces dots with underscores in column names
    # It is required for smooth upload of data into BigQuery down the road
    df = df.toDF(*(c.replace('.', '_') for c in df.columns))
    
    # Saving the data to BigQuery
    # 'com.google.cloud.spark.bigquery'
    table = "{}.{}.{}".format(gcp_project_id, dataset_name, table_name)
    df.write.format('bigquery') \
        .option("temporaryGcsBucket", temp_gcp_bucket_name) \
        .option('table', table) \
        .mode("overwrite") \
        .save()


CPU times: user 9 µs, sys: 2 µs, total: 11 µs
Wall time: 17.2 µs


In [4]:
%%time
print("Started the workflow")
print("Started the LEI Level 1 Golden-Copy data upload")
csv_file = "gs://lei-data/20210512/20210512-0000-gleif-goldencopy-lei2-golden-copy.csv"

gcp_project_id = "lorelee-keep-312708"
dataset_name = "lei_staging"
table_name = "entities"

temp_gcp_bucket_name = "temp-dataproc-storage"

load_csv_to_bigquery (
    csv_file,
    gcp_project_id, 
    dataset_name, 
    table_name,
    temp_gcp_bucket_name
)
print("Finished the LEI Level 1 Golden-Copy data upload")

Started the workflow
Started the LEI Level 1 Golden-Copy data upload
Finished the LEI Level 1 Golden-Copy data upload
CPU times: user 72 ms, sys: 23.9 ms, total: 95.9 ms
Wall time: 1min 51s


In [5]:
%%time
print("Started the LEI Level 2 Golden-Copy data upload")
csv_file = "gs://lei-data/20210513/20210513-0800-gleif-goldencopy-rr-golden-copy.csv"

gcp_project_id = "lorelee-keep-312708"
dataset_name = "lei_staging"
table_name = "ownership"

temp_gcp_bucket_name = "temp-dataproc-storage"

load_csv_to_bigquery (
    csv_file,
    gcp_project_id, 
    dataset_name, 
    table_name,
    temp_gcp_bucket_name
)
print("Finished the LEI Level 2 Golden-Copy data upload")

Started the LEI Level 2 Golden-Copy data upload
Finished the LEI Level 2 Golden-Copy data upload
CPU times: user 27.7 ms, sys: 1.42 ms, total: 29.1 ms
Wall time: 15.8 s


In [6]:
%%time
print("Started the LEI Reporting Exceptions Golden-Copy data upload")
csv_file = "gs://lei-data/20210513/20210513-0800-gleif-goldencopy-repex-golden-copy.csv"

gcp_project_id = "lorelee-keep-312708"
dataset_name = "lei_staging"
table_name = "exceptions"

temp_gcp_bucket_name = "temp-dataproc-storage"

load_csv_to_bigquery (
    csv_file,
    gcp_project_id, 
    dataset_name, 
    table_name,
    temp_gcp_bucket_name
)
print("Finished the LEI Reporting Exceptions Golden-Copy data upload")

Started the LEI Reporting Exceptions Golden-Copy data upload
Finished the LEI Reporting Exceptions Golden-Copy data upload
CPU times: user 18.5 ms, sys: 12 ms, total: 30.4 ms
Wall time: 34.2 s


In [7]:
print("Finished the workflow")

Finished the workflow
